# Experimental Evaluation

This notebook evaluates the outputs produced by `execution.ipynb` without rerunning the LLM-based pipeline.
It compares the original top-down pipeline with the extended pipeline obtained after the two new blocks: bottom-up reconstruction and inter-level feedback.

The notebook computes:

- precision, recall and soft F1 for the original pipeline;
- diagnostic measures for the intermediate bottom-up reconstruction;
- precision, recall and soft F1 for the final goals produced after the feedback loop;
- direct differences between baseline and extended-pipeline metrics;
- CSV tables for reproducible comparison with the results reported in the reference paper.


## 1. Environment and evaluation configuration

The evaluation reads the JSON files generated by the execution notebook. No LLM call is performed in the main evaluation workflow.

In [ ]:
from collections import Counter
from pathlib import Path
import csv
import json
import re

import matplotlib.pyplot as plt
import numpy as np

from groundtruth import (
    GENOME,
    GESTAO_HOSPITAL,
    SIA_PROJECT_25_26,
    LONDON_AMBULANCE_SYSTEM,
)
from src.evaluation.goal_evaluator import GoalEvaluator
from src.examples.shot_learning import ShotPromptingMode

OUTPUT_DIR = Path("output")
EVALUATION_DIR = OUTPUT_DIR / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

PREPROCESSING = True
PROMPTING_MODES = [
    ShotPromptingMode.ZERO_SHOT,
    ShotPromptingMode.ONE_SHOT,
    ShotPromptingMode.FEW_SHOT,
]
LLAMA_ABLATIONS = [False, True]

# Each dataset must occur only once.
GROUNDTRUTHS = [
    GENOME,
    GESTAO_HOSPITAL,
    LONDON_AMBULANCE_SYSTEM,
    SIA_PROJECT_25_26,
]

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Datasets: {[gt['name'] for gt in GROUNDTRUTHS]}")

## 2. Utility functions

The following helpers normalize the heterogeneous JSON structures used by the pipeline and load one output file for each dataset and experimental configuration.

In [ ]:
def item_to_text(item) -> str:
    """Converts strings or serialized Pydantic objects into evaluable text."""
    if item is None:
        return ""
    if isinstance(item, str):
        return item.strip()
    if isinstance(item, dict):
        preferred_keys = (
            "name",
            "reconstructed_high_level_goal",
            "description",
            "goal",
            "text",
        )
        parts = []
        for key in preferred_keys:
            value = item.get(key)
            if isinstance(value, str) and value.strip():
                parts.append(value.strip())
        if parts:
            return ": ".join(dict.fromkeys(parts))
        return json.dumps(item, ensure_ascii=False, sort_keys=True)
    return str(item).strip()


def collection_to_texts(value) -> list[str]:
    """Normalizes lists, dictionaries and scalar values to a list of strings."""
    if value is None:
        return []
    if isinstance(value, dict):
        iterable = value.values()
    elif isinstance(value, (list, tuple, set)):
        iterable = value
    else:
        iterable = [value]

    texts = [item_to_text(item) for item in iterable]
    return [text for text in texts if text]


def output_filename(dataset_name: str, prompting_mode, llama_ablation: bool) -> Path:
    suffix = "_noLlama" if llama_ablation else ""
    return OUTPUT_DIR / f"{dataset_name}_{prompting_mode.name}{suffix}.json"


def load_configuration(prompting_mode, llama_ablation: bool) -> dict[str, dict]:
    loaded = {}
    missing = []

    for groundtruth in GROUNDTRUTHS:
        dataset_name = groundtruth["name"]
        path = output_filename(dataset_name, prompting_mode, llama_ablation)

        if not path.exists():
            missing.append(str(path))
            continue

        with path.open("r", encoding="utf-8") as file:
            loaded[dataset_name] = json.load(file)

    if missing:
        print("Missing files for this configuration:")
        for path in missing:
            print(f"  - {path}")

    return loaded


def configuration_name(prompting_mode, llama_ablation: bool) -> str:
    return f"{prompting_mode.name}{'_noLlama' if llama_ablation else ''}"

## 3. Reference data

The manually defined actors, high-level goals and low-level goals are used as the reference sets for the baseline evaluation.

In [ ]:
manual_actors = {
    gt["name"]: collection_to_texts(gt["actors"])
    for gt in GROUNDTRUTHS
}
manual_high_level_goals = {
    gt["name"]: collection_to_texts(gt["highLevelGoals"])
    for gt in GROUNDTRUTHS
}
manual_low_level_goals = {
    gt["name"]: collection_to_texts(gt["lowLevelGoals"])
    for gt in GROUNDTRUTHS
}

for dataset_name in manual_actors:
    print(
        f"{dataset_name}: "
        f"{len(manual_actors[dataset_name])} actors, "
        f"{len(manual_high_level_goals[dataset_name])} HLGs, "
        f"{len(manual_low_level_goals[dataset_name])} LLGs"
    )

## 4. Metric computation

`GoalEvaluator` creates contextual embeddings and computes a one-to-one optimal matching between generated and reference elements. The resulting similarities are used to derive soft precision, recall and F1.

In [ ]:
def evaluate_goal_sets(
    generated_by_dataset: dict[str, list[str]],
    reference_by_dataset: dict[str, list[str]],
    evaluator: GoalEvaluator,
) -> tuple[list[dict], dict]:
    rows = []
    weighted_precision = 0.0
    weighted_recall = 0.0
    total_weight = 0

    for dataset_name, references in reference_by_dataset.items():
        generated = generated_by_dataset.get(dataset_name, [])
        result = evaluator.calculate_soft_f1(generated, references)

        if isinstance(result, float):
            precision = recall = f1 = result
        else:
            precision = float(result["precision"])
            recall = float(result["recall"])
            f1 = float(result["f1_score"])

        weight = len(references)
        weighted_precision += precision * weight
        weighted_recall += recall * weight
        total_weight += weight

        rows.append({
            "dataset": dataset_name,
            "generated": len(generated),
            "reference": len(references),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

    if total_weight:
        precision = weighted_precision / total_weight
        recall = weighted_recall / total_weight
    else:
        precision = recall = 0.0

    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

    summary = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "reference_weight": total_weight,
    }
    return rows, summary


def print_metric_table(rows: list[dict], summary: dict) -> None:
    header = f"{'Dataset':30} {'Gen':>5} {'Ref':>5} {'Precision':>10} {'Recall':>10} {'F1':>10}"
    print(header)
    print("-" * len(header))
    for row in rows:
        print(
            f"{row['dataset'][:30]:30} {row['generated']:5d} {row['reference']:5d} "
            f"{row['precision']:10.4f} {row['recall']:10.4f} {row['f1']:10.4f}"
        )
    print("-" * len(header))
    print(
        f"{'Weighted average':30} {'':5} {summary['reference_weight']:5d} "
        f"{summary['precision']:10.4f} {summary['recall']:10.4f} {summary['f1']:10.4f}"
    )

# Part I — Baseline top-down evaluation

This part evaluates the outputs of the original architecture: actor extraction, high-level goal extraction and low-level goal decomposition.

## 5. Load baseline outputs

In [ ]:
baseline_outputs = {}

for prompting_mode in PROMPTING_MODES:
    for llama_ablation in LLAMA_ABLATIONS:
        config = configuration_name(prompting_mode, llama_ablation)
        baseline_outputs[config] = load_configuration(prompting_mode, llama_ablation)
        print(f"{config}: loaded {len(baseline_outputs[config])} dataset(s)")

## 6. Actors, high-level goals and low-level goals

The cell evaluates every available prompting/ablation configuration and stores a compact comparative summary.

In [ ]:
evaluator_actors = GoalEvaluator(preprocess=False)
evaluator_goals = GoalEvaluator(preprocess=PREPROCESSING)

baseline_summary_rows = []
baseline_detail = {}

for config, outputs_by_dataset in baseline_outputs.items():
    if not outputs_by_dataset:
        continue

    generated_actors = {
        name: collection_to_texts(output.get("actors"))
        for name, output in outputs_by_dataset.items()
    }
    generated_hl = {
        name: collection_to_texts(output.get("highLevelGoals"))
        for name, output in outputs_by_dataset.items()
    }
    generated_ll = {
        name: collection_to_texts(output.get("lowLevelGoals"))
        for name, output in outputs_by_dataset.items()
    }

    actor_rows, actor_summary = evaluate_goal_sets(
        generated_actors, manual_actors, evaluator_actors
    )
    hl_rows, hl_summary = evaluate_goal_sets(
        generated_hl, manual_high_level_goals, evaluator_goals
    )
    ll_rows, ll_summary = evaluate_goal_sets(
        generated_ll, manual_low_level_goals, evaluator_goals
    )

    baseline_detail[config] = {
        "actors": actor_rows,
        "high_level_goals": hl_rows,
        "low_level_goals": ll_rows,
    }

    for artifact, summary in (
        ("actors", actor_summary),
        ("high_level_goals", hl_summary),
        ("low_level_goals", ll_summary),
    ):
        baseline_summary_rows.append({
            "configuration": config,
            "artifact": artifact,
            **summary,
        })

for row in baseline_summary_rows:
    print(
        f"{row['configuration']:22} | {row['artifact']:20} | "
        f"P={row['precision']:.4f} R={row['recall']:.4f} F1={row['f1']:.4f}"
    )

## 7. Save baseline metrics

In [ ]:
baseline_csv = EVALUATION_DIR / "baseline_metrics.csv"

with baseline_csv.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "configuration",
            "artifact",
            "precision",
            "recall",
            "f1",
            "reference_weight",
        ],
    )
    writer.writeheader()
    writer.writerows(baseline_summary_rows)

print(f"Saved: {baseline_csv}")

# Part II — Bottom-up extension evaluation

The proposed extension reconstructs one candidate high-level goal from the low-level goals of each branch. Its evaluation is branch-oriented and therefore differs from the baseline set-level evaluation.

## 8. Extract reconstructed and original parent goals

The reconstructed goal is compared with the original parent stored in `branchTraceability`. This measures semantic preservation across the top-down decomposition and subsequent bottom-up reconstruction.

In [ ]:
def extract_reconstructed_text(branch_payload: dict) -> str:
    return item_to_text(branch_payload.get("reconstructed_high_level_goal"))


def extract_original_parent_text(traceability_payload: dict) -> str:
    parent = traceability_payload.get("original_high_level_goal", {})
    return item_to_text(parent)


def branch_alignment_rows(
    outputs_by_dataset: dict[str, dict],
    evaluator: GoalEvaluator,
) -> list[dict]:
    rows = []

    for dataset_name, output in outputs_by_dataset.items():
        reconstructed = output.get("bottomUpHighLevelGoals", {}) or {}
        traceability = output.get("branchTraceability", {}) or {}
        errors = output.get("bottomUpErrors", {}) or {}
        empty = set(output.get("bottomUpEmptyBranches", []) or [])

        branch_ids = sorted(set(traceability) | set(reconstructed) | set(errors) | empty)

        for branch_id in branch_ids:
            reconstructed_text = extract_reconstructed_text(reconstructed.get(branch_id, {}))
            original_text = extract_original_parent_text(traceability.get(branch_id, {}))

            similarity = None
            if reconstructed_text and original_text:
                matrix = evaluator.compute_similarity(
                    [reconstructed_text],
                    [original_text],
                )
                similarity = float(matrix[0, 0])

            reconstructed_payload = reconstructed.get(branch_id, {}) or {}
            rows.append({
                "dataset": dataset_name,
                "branch_id": branch_id,
                "status": (
                    "error" if branch_id in errors
                    else "empty" if branch_id in empty
                    else "reconstructed" if reconstructed_text
                    else "missing"
                ),
                "similarity": similarity,
                "cohesion": reconstructed_payload.get("cohesion"),
                "confidence": reconstructed_payload.get("confidence"),
                "error": errors.get(branch_id),
                "original_goal": original_text,
                "reconstructed_goal": reconstructed_text,
            })

    return rows

## 9. Semantic alignment results

In [ ]:
bottom_up_rows_by_config = {}
bottom_up_summary_rows = []

for config, outputs_by_dataset in baseline_outputs.items():
    if not outputs_by_dataset:
        continue

    rows = branch_alignment_rows(outputs_by_dataset, evaluator_goals)
    bottom_up_rows_by_config[config] = rows

    similarities = [
        row["similarity"]
        for row in rows
        if row["similarity"] is not None
    ]
    status_counts = Counter(row["status"] for row in rows)
    total_branches = len(rows)
    reconstructed_count = status_counts.get("reconstructed", 0)

    summary = {
        "configuration": config,
        "total_branches": total_branches,
        "reconstructed_branches": reconstructed_count,
        "empty_branches": status_counts.get("empty", 0),
        "error_branches": status_counts.get("error", 0),
        "missing_branches": status_counts.get("missing", 0),
        "coverage": reconstructed_count / total_branches if total_branches else 0.0,
        "mean_parent_similarity": float(np.mean(similarities)) if similarities else 0.0,
        "median_parent_similarity": float(np.median(similarities)) if similarities else 0.0,
    }
    bottom_up_summary_rows.append(summary)

for row in bottom_up_summary_rows:
    print(
        f"{row['configuration']:22} | coverage={row['coverage']:.4f} | "
        f"mean similarity={row['mean_parent_similarity']:.4f} | "
        f"empty={row['empty_branches']} | errors={row['error_branches']}"
    )

## 10. Branch-level inspection

Set `SELECTED_CONFIGURATION` to inspect individual branches, their reconstructed goal, the original parent and the semantic similarity score.

In [ ]:
SELECTED_CONFIGURATION = next(iter(bottom_up_rows_by_config), None)

if SELECTED_CONFIGURATION is None:
    print("No bottom-up result is available.")
else:
    print(f"Configuration: {SELECTED_CONFIGURATION}
")
    for row in bottom_up_rows_by_config[SELECTED_CONFIGURATION]:
        print(f"[{row['dataset']} | {row['branch_id']}] {row['status']}")
        print(f"Original:      {row['original_goal']}")
        print(f"Reconstructed: {row['reconstructed_goal']}")
        print(f"Similarity:    {row['similarity']}")
        print(f"Cohesion:      {row['cohesion']}")
        print(f"Confidence:    {row['confidence']}")
        if row["error"]:
            print(f"Error:         {row['error']}")
        print("-" * 80)

## 11. Cohesion and confidence analysis

The generator returns ordinal judgements (`high`, `medium`, `low`) for branch cohesion and reconstruction confidence. The following cells aggregate and visualize them.

In [ ]:
def ordinal_counts(rows: list[dict], field: str) -> dict[str, int]:
    counts = Counter(
        row[field]
        for row in rows
        if row.get(field) in {"low", "medium", "high"}
    )
    return {level: counts.get(level, 0) for level in ("low", "medium", "high")}

for config, rows in bottom_up_rows_by_config.items():
    print(config)
    print("  cohesion:  ", ordinal_counts(rows, "cohesion"))
    print("  confidence:", ordinal_counts(rows, "confidence"))

In [ ]:
if SELECTED_CONFIGURATION is not None:
    rows = bottom_up_rows_by_config[SELECTED_CONFIGURATION]

    cohesion = ordinal_counts(rows, "cohesion")
    plt.figure(figsize=(7, 4))
    plt.bar(cohesion.keys(), cohesion.values())
    plt.xlabel("Cohesion")
    plt.ylabel("Number of branches")
    plt.title(f"Cohesion distribution — {SELECTED_CONFIGURATION}")
    plt.show()

    confidence = ordinal_counts(rows, "confidence")
    plt.figure(figsize=(7, 4))
    plt.bar(confidence.keys(), confidence.values())
    plt.xlabel("Confidence")
    plt.ylabel("Number of branches")
    plt.title(f"Confidence distribution — {SELECTED_CONFIGURATION}")
    plt.show()

## 12. Save bottom-up metrics

In [ ]:
bottom_up_summary_csv = EVALUATION_DIR / "bottom_up_summary.csv"
with bottom_up_summary_csv.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(bottom_up_summary_rows[0].keys()) if bottom_up_summary_rows else [])
    if bottom_up_summary_rows:
        writer.writeheader()
        writer.writerows(bottom_up_summary_rows)

bottom_up_branches_csv = EVALUATION_DIR / "bottom_up_branch_details.csv"
all_branch_rows = []
for config, rows in bottom_up_rows_by_config.items():
    for row in rows:
        all_branch_rows.append({"configuration": config, **row})

with bottom_up_branches_csv.open("w", newline="", encoding="utf-8") as file:
    if all_branch_rows:
        writer = csv.DictWriter(file, fieldnames=list(all_branch_rows[0].keys()))
        writer.writeheader()
        writer.writerows(all_branch_rows)

print(f"Saved: {bottom_up_summary_csv}")
print(f"Saved: {bottom_up_branches_csv}")

# Part III — Final extended-pipeline evaluation

This section evaluates the **final high-level and low-level goals after the inter-level feedback loop**.
The reconstructed bottom-up goal is an intermediate diagnostic artifact and is not used as a direct replacement for the final goal set.

The execution notebook should persist the final corrected outputs using the keys `finalHighLevelGoals` and `finalLowLevelGoals`.
When these keys are not available, the corresponding configuration is skipped rather than incorrectly treating the intermediate reconstruction as a final output.


In [ ]:
FINAL_HL_KEY = "finalHighLevelGoals"
FINAL_LL_KEY = "finalLowLevelGoals"

extended_summary_rows = []
extended_detail = {}

for config, outputs_by_dataset in baseline_outputs.items():
    if not outputs_by_dataset:
        continue

    available_outputs = {
        dataset_name: output
        for dataset_name, output in outputs_by_dataset.items()
        if FINAL_HL_KEY in output and FINAL_LL_KEY in output
    }

    if not available_outputs:
        print(
            f"{config}: skipped final evaluation because "
            f"'{FINAL_HL_KEY}' and '{FINAL_LL_KEY}' are not available."
        )
        continue

    generated_final_hl = {
        name: collection_to_texts(output.get(FINAL_HL_KEY))
        for name, output in available_outputs.items()
    }
    generated_final_ll = {
        name: collection_to_texts(output.get(FINAL_LL_KEY))
        for name, output in available_outputs.items()
    }

    reference_hl = {
        name: manual_high_level_goals[name]
        for name in available_outputs
    }
    reference_ll = {
        name: manual_low_level_goals[name]
        for name in available_outputs
    }

    hl_rows, hl_summary = evaluate_goal_sets(
        generated_final_hl, reference_hl, evaluator_goals
    )
    ll_rows, ll_summary = evaluate_goal_sets(
        generated_final_ll, reference_ll, evaluator_goals
    )

    extended_detail[config] = {
        "high_level_goals": hl_rows,
        "low_level_goals": ll_rows,
    }

    for artifact, summary in (
        ("high_level_goals", hl_summary),
        ("low_level_goals", ll_summary),
    ):
        extended_summary_rows.append({
            "configuration": config,
            "artifact": artifact,
            **summary,
        })

for row in extended_summary_rows:
    print(
        f"{row['configuration']:22} | FINAL {row['artifact']:20} | "
        f"P={row['precision']:.4f} R={row['recall']:.4f} F1={row['f1']:.4f}"
    )


## 13. Save final extended-pipeline metrics

The following table stores the final precision, recall and F1 values used for the direct comparison with the baseline.


In [ ]:
extended_csv = EVALUATION_DIR / "extended_pipeline_metrics.csv"

with extended_csv.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "configuration",
            "artifact",
            "precision",
            "recall",
            "f1",
            "reference_weight",
        ],
    )
    writer.writeheader()
    writer.writerows(extended_summary_rows)

print(f"Saved: {extended_csv}")


# Part IV — Baseline versus extended pipeline

The final comparison uses the same precision, recall and soft F1 procedure for both configurations.
The table reports the baseline metrics, the final post-feedback metrics and their differences.
Bottom-up coverage and reconstructed-parent similarity remain diagnostic measures and are not substitutes for final precision, recall or F1.


In [ ]:
comparison = {}

for row in baseline_summary_rows:
    config = row["configuration"]
    artifact = row["artifact"]
    comparison.setdefault(config, {"configuration": config})
    comparison[config][f"baseline_{artifact}_precision"] = row["precision"]
    comparison[config][f"baseline_{artifact}_recall"] = row["recall"]
    comparison[config][f"baseline_{artifact}_f1"] = row["f1"]

for row in extended_summary_rows:
    config = row["configuration"]
    artifact = row["artifact"]
    comparison.setdefault(config, {"configuration": config})
    comparison[config][f"extended_{artifact}_precision"] = row["precision"]
    comparison[config][f"extended_{artifact}_recall"] = row["recall"]
    comparison[config][f"extended_{artifact}_f1"] = row["f1"]

for config, row in comparison.items():
    for artifact in ("high_level_goals", "low_level_goals"):
        for metric in ("precision", "recall", "f1"):
            baseline_key = f"baseline_{artifact}_{metric}"
            extended_key = f"extended_{artifact}_{metric}"
            if baseline_key in row and extended_key in row:
                row[f"delta_{artifact}_{metric}"] = (
                    row[extended_key] - row[baseline_key]
                )

for row in bottom_up_summary_rows:
    config = row["configuration"]
    comparison.setdefault(config, {"configuration": config})
    comparison[config].update({
        "bottom_up_coverage": row["coverage"],
        "bottom_up_mean_similarity": row["mean_parent_similarity"],
        "bottom_up_empty_branches": row["empty_branches"],
        "bottom_up_error_branches": row["error_branches"],
    })

comparison_rows = list(comparison.values())
for row in comparison_rows:
    print(json.dumps(row, ensure_ascii=False, indent=2))


In [ ]:
comparison_csv = EVALUATION_DIR / "comparative_summary.csv"

all_fields = ["configuration"]
for row in comparison_rows:
    for key in row:
        if key not in all_fields:
            all_fields.append(key)

with comparison_csv.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=all_fields)
    writer.writeheader()
    writer.writerows(comparison_rows)

print(f"Saved: {comparison_csv}")

## 14. Interpretation checklist

When reporting the results in the thesis, distinguish clearly between:

1. **Baseline extraction quality**, measured against the same manually defined actors and goals used in the reference paper.
2. **Intermediate bottom-up diagnostics**, measured through reconstruction coverage, failures and similarity with the original parent.
3. **Final extended-pipeline quality**, measured only on the high-level and low-level goals produced after the feedback loop.
4. **Improvement over the baseline**, measured as the difference in precision, recall and F1 under the same datasets, preprocessing and matching procedure.

The reconstructed bottom-up goal is not itself the final output of the pipeline. It supports the feedback process, while the comparison with the paper must use the final corrected goal sets.
